# TernakKu — Modul 1: Eksperimen Estimasi Bobot Sapi (Colab)

Notebook ini menarik kode + data dari GitHub lalu menjalankan **mesin eksperimen yang sama** dengan platform web (`ml/experiment.py`): pilih metode, fitur (termasuk fitur allometrik), target log, dan model hibrida — lalu bandingkan akurasinya.

Juga mereproduksi temuan penting: **model meleset jauh untuk sapi lokal kecil (Bali) tapi akurat untuk sapi besar (Simbal)** — bukti perlunya rekalibrasi data lokal.

Reproducible: versi paket dikunci, seed = 42. **Runtime → Run all** untuk menjalankan semuanya.

Repo: https://github.com/skivanoclaire/ternakku

## 1. Ambil kode + data, pasang paket (versi terkunci)

In [ ]:
!git clone -q https://github.com/skivanoclaire/ternakku.git
%cd ternakku
!pip install -q -r ml/requirements.txt
print('siap')

## 2. Siapkan dataset publik (Hereford + Horqin)

In [ ]:
!python ml/prep_data.py --indir data/raw --outdir data/out
import sys; sys.path.append('ml')
import experiment, joblib, pandas as pd
CSV = 'data/out/pengukuran_public.csv'
OUT = 'data/out'

## 3. Bandingkan metode (leaderboard mini)
Ubah daftar `configs` untuk bereksperimen: `(metode, [fitur], target_log)`.
Metode: `schoorl`, `loglog`, `linear`, `rf`, `xgb`, `hybrid`.
Fitur: `lingkar_dada_cm`, `panjang_badan_cm`, `tinggi_gumba_cm`, `ld2`, `ld2_pb` (volume), `pb_ld`, `tg_ld`.

In [ ]:
RAW = ['lingkar_dada_cm','panjang_badan_cm','tinggi_gumba_cm']
configs = [
    ('schoorl', ['lingkar_dada_cm'],        False),
    ('loglog',  ['lingkar_dada_cm'],        False),
    ('linear',  RAW,                        False),
    ('linear',  RAW + ['ld2_pb'],           True),
    ('rf',      RAW,                        False),
    ('xgb',     RAW,                        False),
    ('hybrid',  RAW + ['ld2_pb'],           False),
]
rows = []
for i,(m,f,lg) in enumerate(configs, 1):
    r = experiment.run(CSV, m, f, 'acak', OUT, i, 'B', lg)
    mt = r['metrics']
    rows.append({'metode': m+('(log)' if lg else ''), 'fitur': '+'.join(f),
                 'MAPE%': mt['mape'], 'MAE': mt['mae'], 'R2': mt['r2']})
pd.DataFrame(rows).sort_values('MAPE%').reset_index(drop=True)

## 4. Uji ke data jurnal sapi lokal vs sapi besar
Latih satu model (XGBoost), lalu prediksi ukuran sapi dari dua jurnal. Lihat: **Simbal (besar) akurat, Bali (lokal kecil) meleset jauh** — karena data latih (Hereford/Horqin) berisi sapi besar.

In [ ]:
r = experiment.run(CSV, 'xgb', RAW, 'acak', OUT, 99, 'B', False)
model = joblib.load(f"{OUT}/experiments/{r['model_ver']}.joblib")

# (nama, LD, PB, TG, bobot asli di jurnal)
kasus = [
    ('Sapi Bali jantan',  140.8, 113.27, 104.73, 205.9),
    ('Sapi Bali betina',  129.2, 104.73, 103.03, 180.3),
    ('Simbal jantan',     160.4, 134.40, 124.93, 365.1),
    ('Simbal betina',     157.3, 129.03, 120.77, 340.6),
    ('Sapi Krui poel 2',  134.5, 124.60, 105.10, 189.6),
]
out = []
for n, ld, pb, tg, truth in kasus:
    p = experiment.predict_one(model, ld, pb, tg)
    out.append({'kasus': n, 'LD': ld, 'paper(kg)': truth,
                'model(kg)': round(p,1), 'error%': round((p-truth)/truth*100)})
pd.DataFrame(out)

## 5. Demonstrasi rekalibrasi lokal
Latih model **log-log khusus data lokal** (rerata dari kedua jurnal) lalu prediksi ulang Sapi Bali — angkanya turun mendekati nyata. **Catatan:** hanya ilustrasi (sampel sangat kecil); produksi butuh banyak data per ekor.

In [ ]:
import numpy as np
# titik data lokal dari jurnal (LD, bobot)
lokal = np.array([
    [118,125.4],[132.9,148.6],[134.5,189.6],[137,234.5],   # Krui
    [140.8,205.9],[129.15,180.3],[160.37,365.1],[157.3,340.6],  # Bali & Simbal
])
ld_l, bb_l = lokal[:,0], lokal[:,1]
b, a = np.polyfit(np.log(ld_l), np.log(bb_l), 1)   # log-log lokal
pred_bali = float(np.exp(a + b*np.log(140.8)))
print(f'Model lokal (log-log): eksponen b={b:.2f}')
print(f'Prediksi Sapi Bali jantan (LD 140,8): {pred_bali:.0f} kg  (jurnal: 205,9 kg)')
print('Bandingkan dengan model lama (sapi besar): 425 kg — rekalibrasi lokal jauh lebih dekat.')

## 6. (Opsional) Visualisasi LD vs bobot

In [ ]:
import matplotlib.pyplot as plt
df = pd.read_csv(CSV)
for nm, g in df.groupby('src_dataset'):
    plt.scatter(g['lingkar_dada_cm'], g['bobot_timbang_kg'], s=8, alpha=.4, label='latih: '+str(nm))
plt.scatter(lokal[:,0], lokal[:,1], c='red', s=60, marker='x', label='sapi lokal (jurnal)')
plt.xlabel('Lingkar dada (cm)'); plt.ylabel('Bobot (kg)'); plt.legend(); plt.title('Data latih vs sapi lokal')
plt.show()